# Customer Churn Prediction
### Optimised for Recall — catching as many churners as possible

## Cell 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, recall_score, precision_score
import pickle
import json

# recall_score and precision_score added so we can directly measure them
print('All libraries imported successfully!')

## Cell 2 — Load the Dataset

In [ ]:
df = pd.read_excel('data/Telco_customer_churn.xlsx')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

## Cell 3 — Drop Unnecessary Columns

In [ ]:
df.drop([
    'CustomerID',
    'Count',
    'Country',
    'State',
    'City',
    'Zip Code',
    'Lat Long',
    'Latitude',
    'Longitude',
    'Churn Label',
    'Churn Score',
    'Churn Reason'
], axis=1, inplace=True)

print('Remaining columns:', df.shape[1])
print(df.columns.tolist())

## Cell 4 — Fix Data Types & Missing Values

In [ ]:
df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')

print('Missing values before drop:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df.dropna(inplace=True)

print('\nShape after cleaning:', df.shape)
print('Missing values remaining:', df.isnull().sum().sum())

## Cell 5 — Check Churn Balance

In [ ]:
print('Churn distribution:')
print(df['Churn Value'].value_counts())
print('\nPercentage:')
print(df['Churn Value'].value_counts(normalize=True) * 100)

# This imbalance (73% stayed, 27% churned) is WHY we optimise for recall
# A model that always predicts stayed would get 73% accuracy but catch 0 churners

plt.figure(figsize=(5,4))
df['Churn Value'].value_counts().plot(kind='bar', color=['steelblue','tomato'])
plt.xticks([0,1], ['Stayed', 'Churned'], rotation=0)
plt.title('Churn Distribution — Dataset is Imbalanced')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('churn_distribution.png')
plt.show()
print('Chart saved!')

## Cell 6 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Churn by Contract Type
contract_churn = df.groupby('Contract')['Churn Value'].mean() * 100
contract_churn.plot(kind='bar', ax=axes[0], color=['steelblue','orange','tomato'])
axes[0].set_title('Churn Rate by Contract Type')
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_xlabel('')
axes[0].tick_params(rotation=15)

# 2. Churn by Monthly Charges
axes[1].boxplot(
    [df[df['Churn Value']==0]['Monthly Charges'],
     df[df['Churn Value']==1]['Monthly Charges']],
    labels=['Stayed', 'Churned']
)
axes[1].set_title('Monthly Charges vs Churn')
axes[1].set_ylabel('Monthly Charges ($)')

# 3. Churn by Tenure
axes[2].hist(df[df['Churn Value']==0]['Tenure Months'], bins=30, alpha=0.6, label='Stayed', color='steelblue')
axes[2].hist(df[df['Churn Value']==1]['Tenure Months'], bins=30, alpha=0.6, label='Churned', color='tomato')
axes[2].set_title('Tenure vs Churn')
axes[2].set_xlabel('Tenure (Months)')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda_charts.png')
plt.show()
print('EDA charts saved!')

## Cell 7 — Encode Categorical Columns

In [ ]:
X = df.drop('Churn Value', axis=1)
y = df['Churn Value']

X = pd.get_dummies(X, drop_first=True)

print('Features shape:', X.shape)
print('\nAll columns are now numeric:')
print(X.dtypes.value_counts())

## Cell 8 — Split Data into Train & Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # NEW — ensures both train and test have same churn ratio
                # without this, by random chance test set might have very few churners
)

print('Training set size:', X_train.shape[0], 'customers')
print('Test set size:', X_test.shape[0], 'customers')
print('\nChurn rate in training set:', round(y_train.mean() * 100, 1), '%')
print('Churn rate in test set:', round(y_test.mean() * 100, 1), '%')

## Cell 9 — Train the Model (Optimised for Recall)

In [ ]:
# class_weight='balanced' is the key change for recall optimisation
# It tells the model: churners are rare so penalise missing them MORE
# Internally it multiplies the importance of churner errors by (5174/1869) = ~2.8x
# So missing a churner is treated as 2.8x worse than misclassifying someone who stayed

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'  # NEW — this is what makes it recall-focused
)

model.fit(X_train, y_train)

print('Model trained successfully with balanced class weights!')

## Cell 10 — Default Threshold Evaluation (0.5)

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

# Default threshold = 0.5 (predict churn if probability > 50%)
y_pred_default = model.predict(X_test)

print('=== Default Threshold (0.5) ===')
print(classification_report(y_test, y_pred_default, target_names=['Stayed', 'Churned']))
print('AUC-ROC:', round(roc_auc_score(y_test, y_prob), 4))

## Cell 11 — Lower Threshold to Improve Recall

In [ ]:
# Instead of predicting churn only when probability > 50%
# We lower it to 30% — so the model flags more people as churners
# This catches more real churners (higher recall)
# But also flags more false alarms (lower precision) — acceptable tradeoff

THRESHOLD = 0.30

y_pred_adjusted = (y_prob >= THRESHOLD).astype(int)

print(f'=== Adjusted Threshold ({THRESHOLD}) ===')
print(classification_report(y_test, y_pred_adjusted, target_names=['Stayed', 'Churned']))

recall = recall_score(y_test, y_pred_adjusted)
precision = precision_score(y_test, y_pred_adjusted)
print(f'Churner Recall:    {round(recall * 100, 1)}%  — of all real churners, we caught this many')
print(f'Churner Precision: {round(precision * 100, 1)}%  — of flagged churners, this many actually churned')

## Cell 12 — Compare All Thresholds Visually

In [ ]:
# This shows how precision and recall change as you move the threshold
# You can use this chart to pick the best threshold for your business

thresholds = np.arange(0.1, 0.9, 0.05)
recalls = []
precisions = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))

plt.figure(figsize=(10, 5))
plt.plot(thresholds, recalls, label='Recall (catching churners)', color='tomato', linewidth=2)
plt.plot(thresholds, precisions, label='Precision (accuracy of flags)', color='steelblue', linewidth=2)
plt.axvline(x=0.30, color='green', linestyle='--', label='Our chosen threshold (0.30)')
plt.axvline(x=0.50, color='gray', linestyle='--', label='Default threshold (0.50)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs Recall at Different Thresholds')
plt.legend()
plt.tight_layout()
plt.savefig('precision_recall_tradeoff.png', dpi=150)
plt.show()
print('Chart saved! This is a great chart to include in your LinkedIn post.')

## Cell 13 — Confusion Matrix

In [ ]:
# Confusion matrix shows exactly where the model is right and wrong
# Rows = actual, Columns = predicted
# Top left  = correctly predicted stayed (True Negative)
# Top right = wrongly predicted churn when they stayed (False Positive)
# Bottom left  = wrongly predicted stayed when they churned (False Negative) — THE COSTLY ONE
# Bottom right = correctly predicted churn (True Positive) — WHAT WE WANT TO MAXIMISE

cm = confusion_matrix(y_test, y_pred_adjusted)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Stayed', 'Predicted Churned'],
            yticklabels=['Actually Stayed', 'Actually Churned'])
plt.title(f'Confusion Matrix (Threshold = 0.30)')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'Correctly caught churners (True Positives):  {tp}')
print(f'Missed churners (False Negatives):           {fn}  ← we want this LOW')
print(f'False alarms (False Positives):              {fp}')
print(f'Correctly identified loyal customers:        {tn}')

## Cell 14 — Feature Importance Chart

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
top10 = importances.nlargest(10).sort_values()

plt.figure(figsize=(8, 5))
top10.plot(kind='barh', color='steelblue')
plt.title('Top 10 Factors Driving Customer Churn', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print('Feature importance chart saved!')

## Cell 15 — Save the Model and Threshold

In [ ]:
# Save model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save column names
with open('columns.json', 'w') as f:
    json.dump(list(X.columns), f)

# Save threshold so app uses same value
with open('threshold.json', 'w') as f:
    json.dump({'threshold': THRESHOLD}, f)

print('Model saved as model.pkl')
print('Columns saved as columns.json')
print('Threshold saved as threshold.json')
print('\nAll done! Your model is now optimised for recall.')